<h1>Importing Libraries</h1>

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler, StandardScaler, KBinsDiscretizer
from scipy.stats import zscore
import matplotlib.pyplot as plt
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
df = pd.read_csv(r"D:\AI Course Stuff\archive\Pakistan Largest Ecommerce Dataset.csv")
df

<h1>Checking Columns' Names</h1>

In [ ]:
df.columns

<h1>Checking for NaN values in the data</h1>

In [ ]:
df.isnull().sum()

<h1>Dropping Unnecessary Columns with All NaN values</h1>

In [ ]:
cols = ['Unnamed: 21','Unnamed: 22','Unnamed: 23','Unnamed: 24','Unnamed: 25']
df_new = df.drop(columns=cols)
print(df_new)

<h1>Checking if there is any single valid/non-NaN value and dropping all those rows in which all of the values are NaN</h1>

In [ ]:
df1 = pd.DataFrame(df_new)
df1.dropna(thresh=1, inplace=True)
df1

<h1>Checking the NaN values in data again</h1>

In [ ]:
df1.isnull().sum()

<h1>Checking the remaining columns</h1>

In [ ]:
df1.columns

In [ ]:
cols2 = ['sales_commission_code','increment_id','Working Date',' MV ','Year','Month','Customer Since','M-Y','FY','created_at']
df2 = df1.drop(columns=cols2)
print(df2)

<h1>Forward Filling Missing Values in the columns</h1>

In [ ]:
df2['status']=df2['status'].fillna(method='ffill')
df2['sku']=df2['sku'].fillna(method='ffill')
df2['category_name_1']=df2['category_name_1'].fillna(method='ffill')
df2['Customer ID']=df2['Customer ID'].fillna(method='ffill')
print(df2)

<h1>Checking for missing values again</h1>

In [ ]:
df2.isnull().sum()

<h1>Label Encoding Categorical Columns</h1>

In [ ]:
label_encoder = LabelEncoder()
df2['status'] = label_encoder.fit_transform(df2['status'])
df2['sku'] = label_encoder.fit_transform(df2['sku'])
df2['category_name_1'] = label_encoder.fit_transform(df2['category_name_1'])
df2['BI Status'] = label_encoder.fit_transform(df2['BI Status'])
df2['payment_method'] = label_encoder.fit_transform(df2['payment_method'])
print("\nAfter Label Encoding:")
print(df2)

<h1>Applying Standardization to the required columns</h1>

In [ ]:
scaler = StandardScaler()
df2['price'] = scaler.fit_transform(df2[['price']].abs())
print(df2['price'])
df2['qty_ordered'] = scaler.fit_transform(df2[['qty_ordered']].abs())
print(df2['qty_ordered'])
df2['grand_total'] = scaler.fit_transform(df2[['grand_total']].abs())
print(df2['grand_total'])

<h1>Checking the DataFrame</h1>

In [ ]:
print(df2)

<h1>Dealing with outliers using Z-Score</h1>

In [ ]:
from scipy.stats import zscore
df2['ZScore'] = zscore(df2['price'])
outlier = df2['price'][np.abs(df2['ZScore'])>3]
print(outlier)
df2['ZScore'] = zscore(df2['qty_ordered'])
outlier = df2['qty_ordered'][np.abs(df2['ZScore'])>3]
print(outlier)
outlier_count = len(outlier)
print("Number of outliers:", outlier_count)
df2['ZScore'] = zscore(df2['grand_total'])

outlier = df2['grand_total'][np.abs(df2['ZScore'])>3]
print(outlier)
df2.loc[np.abs(df2['ZScore']) > 3, 'grand_total'] = df2['grand_total'].median()
df2.loc[np.abs(df2['ZScore']) > 3, 'price'] = df2['price'].median()
df2.loc[np.abs(df2['ZScore']) > 3, 'qty_ordered'] = df2['qty_ordered'].median()
print(df2)

<h1>Checking columns again</h1>

In [ ]:
df2.columns

<h1>Printing the DataFrame again</h1>

In [ ]:
print(df2)

<h1>Installing XGBoost</h1>

In [ ]:
pip install xgboost

<h1>Training the Model and Predicting the Accuracy of the Model</h1>

In [ ]:
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report
cols4 = ['category_name_1','item_id','payment_method','BI Status','Customer ID','ZScore']
X = df2.drop(cols4, axis=1) 
y = df2['category_name_1']
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
import xgboost as xgb

model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=10,
    learning_rate=0.2,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

<h1>Fixing Negative Values for Visualization</h1>

In [ ]:
cols_to_fix = ['price', 'qty_ordered', 'grand_total']

df2[cols_to_fix] = df2[cols_to_fix].abs()

<h1>Visualizing All Categories Against Different Factors to Determine Best Selling Category</h1>

In [ ]:
import plotly.express as px

try:
    df2['category_name_1'] = label_encoder.inverse_transform(df2['category_name_1'])
except:
    pass

fig1 = px.bar(
    df2.groupby('category_name_1', as_index=False)['qty_ordered'].sum(),
    x='category_name_1',
    y='qty_ordered',
    title='Top Selling Categories (Quantity Ordered)',
)
fig1.show()

fig2 = px.bar(
    df2.groupby('category_name_1', as_index=False)['grand_total'].sum(),
    x='category_name_1',
    y='grand_total',
    title='Revenue by Category',
)
fig2.show()

fig3 = px.box(
    df2,
    x='category_name_1',
    y='price',
    title='Price Distribution per Category'
)
fig3.show()

fig4 = px.histogram(
    df2,
    x='category_name_1',
    color='status',
    barmode='group',
    title='Order Status Distribution per Category'
)
fig4.show()

fig5 = px.histogram(
    df2,
    x='category_name_1',
    color='payment_method',
    barmode='group',
    title='Payment Method Distribution per Category'
)
fig5.show()

fig6 = px.scatter(
    df2,
    x='discount_amount',
    y='qty_ordered',
    color='category_name_1',
    title='Discount vs Quantity Ordered'
)
fig6.show()

fig7 = px.bar(
    df2.groupby('category_name_1', as_index=False)['price'].mean(),
    x='category_name_1',
    y='price',
    title='Average Price per Category'
)
fig7.show()

fig8 = px.histogram(
    df2,
    x='category_name_1',
    title='Number of Orders per Category'
)
fig8.show()